# Data Download - F1 Strategy Manager

**Objective:** Download data from 46 Grand Prix races (2023-2024) from FastF1 and OpenF1 APIs.

**Data sources:**
- `FastF1`: laps, weather, pit stops, race control messages
- `OpenF1`: Intervals between cars

**Output structure:**
```
data/raw/
├── 2023/
│   ├── Bahrain/
│   │   ├── laps.parquet
│   │   ├── weather.parquet
│   │   ├── intervals.parquet
│   │   ├── pitstops.parquet
│   │   └── metadata.json
│   └── ...
└── 2024/
    └── ...
```

**Calculated basic features:**

*Laps:*
- CompoundID (1=Soft, 2=Medium, 3=Hard, 4=Inter, 5=Wet)
- TeamID (team name to integer mapping)
- LapsSincePitStop (counter since last pit stop)
- FuelLoad (estimation: 1 - LapNumber/MaxLap)

*Intervals:*
- interval_seconds, gap_to_leader_seconds (numeric gaps)
- laps_behind, gap_to_leader_laps (lap deficit strings: "+1 LAP", "+2 LAPS")
- drs_window (within 1.0s DRS activation range, universal FIA threshold)
- is_lapped (boolean flag)

**Note:** `undercut_window` deferred to Phase 2.4 Feature Engineering because pit lane time loss varies significantly by circuit.

**References to legacy notebooks:**
- Legacy extraction: [fastf1_extractor.py](../../src/shared/data_extraction/fastf1_extractor.py)
- Legacy extraction: [openf1_extractor.py](../../src/shared/data_extraction/openf1_extractor.py)
- Legacy data processing: [lap_prediction.ipynb](../../legacy/notebooks/ML_tyre_pred/lap_prediction.ipynb)

---

## Step 2: Imports and Setups

In [21]:
# Imports
import fastf1 as ff1
import pandas as pd
import numpy as np
import requests
from pathlib import Path
import json
from datetime import datetime
from typing import Tuple, Optional
from tqdm.notebook import tqdm


In [22]:
# Find repository root (where .git folder is)
current_path = Path.cwd()
while not (current_path / ".git").exists() and current_path != current_path.parent:
    current_path = current_path.parent
REPO_ROOT = current_path

In [23]:
# Configure absolute paths from repository root
CACHE_DIR = REPO_ROOT / "data" / "cache" / "fastf1"
RAW_DATA_PATH = REPO_ROOT / "data" / "raw"

# Create directories
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Enable FastF1 cache
ff1.Cache.enable_cache(str(CACHE_DIR))

print(f"Repository root: {REPO_ROOT}")
print(f"FastF1 cache: {CACHE_DIR}")
print(f"Raw data path: {RAW_DATA_PATH}")

Repository root: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
FastF1 cache: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\cache\fastf1
Raw data path: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw


---

## Step 3: Configuration Section

In this secion, we'll define the list of all the Grand Prix races of 2023/24 seasons and mappings from calculated features.

**Total races**: 46 GPs, extracted from FastF1 method `get_event_schedule()`. We then use the `Location` key so we get the names of the calendar that FastF1 uses for `get_session()` method that we'll use later on.

- 2023: 22 races.
- 2024: 24 races.

**Important mappings already done in legacy notebooks**: 
- Compound: tire compound to integer ID.
- Team: team name to integer ID.

In [24]:
# Get race calendars dynamically from FastF1
print("Fetching race calendars from FastF1...")

# Get 2023 and 2024 schedules
schedule_2023 = ff1.get_event_schedule(2023)
schedule_2024 = ff1.get_event_schedule(2024)

# Filter only race events (RoundNumber > 0 excludes pre-season testing)
races_2023_df = schedule_2023[schedule_2023['RoundNumber'] > 0]
races_2024_df = schedule_2024[schedule_2024['RoundNumber'] > 0]

# Extract race location names (this is what FastF1 uses for get_session)
RACES_2023 = races_2023_df['Location'].tolist()
RACES_2024 = races_2024_df['Location'].tolist()

Fetching race calendars from FastF1...


In [25]:
# Compound mapping
COMPOUND_MAPPING = {
    'SOFT': 1,
    'MEDIUM': 2,
    'HARD': 3,
    'INTERMEDIATE': 4,
    'WET': 5,
    'TEST_UNKNOWN': 0,
    'UNKNOWN': 0
}

# Team mapping (2023-2024 grid)
TEAM_MAPPING = {
    'Red Bull Racing': 1,
    'Mercedes': 2,
    'Ferrari': 3,
    'McLaren': 4,
    'Aston Martin': 5,
    'Alpine': 6,
    'Williams': 7,
    'AlphaTauri': 8,
    'RB': 8,
    'Alfa Romeo': 9,
    'Sauber': 9,
    'Haas F1 Team': 10
}

# Circuit name aliases — maps FastF1 location names to a single canonical name.
# 'Spain' is a test artefact: Step 4 calls download_fastf1_data(2023, 'Spain'),
# which FastF1 resolves to 'Barcelona' internally but saves under the folder name
# passed here. Applying this alias before save_race_data() ensures all downstream
# notebooks see only 'Barcelona' for the Spanish GP across both seasons.
CIRCUIT_NAME_ALIASES: dict[str, str] = {
    'Spain': 'Barcelona',
}

In [26]:
print(f"\nTotal GPs configured: {len(RACES_2023) + len(RACES_2024)}")
print(f"  2023: {len(RACES_2023)} races")
print(f"  2024: {len(RACES_2024)} races")
print(f"\n2023 races: {RACES_2023[:3]}... (showing first 3)")
print(f"2024 races: {RACES_2024[:3]}... (showing first 3)")


Total GPs configured: 46
  2023: 22 races
  2024: 24 races

2023 races: ['Sakhir', 'Jeddah', 'Melbourne']... (showing first 3)
2024 races: ['Sakhir', 'Jeddah', 'Melbourne']... (showing first 3)


---

## Step 4: Utility Functions Section

Define helper functions for downloading, processing, and saving race data.

**Functions:**
1. `download_fastf1_data()` - Download laps, weather, pitstops from FastF1
2. `calculate_basic_features()` - Add CompoundID, TeamID, LapsSincePitStop, FuelLoad
3. `find_openf1_session_key()` - Find OpenF1 session_key for a race
4. `download_openf1_intervals()` - Download intervals data from OpenF1
5. `save_race_data()` - Save all data to parquet files
6. `validate_race_data()` - Validate saved data completeness


### 4.1 Download FastF1 Data

This function downloads the data we want from FastF1, regarding:
- Laps.
- Weather.
- Pitstops.
Each one is returned as a pandas dataframe. To use the function, we need to pass as arguments the season and GP we want to extract data from.

In [27]:
def download_fastf1_data(year: int, gp_name: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Download laps, weather, and pitstops data from FastF1.
    
    Parameters:
    - year: int (2023 or 2024)
    - gp_name: str (e.g., 'Bahrain', 'Spain')
    
    Returns:
    - laps_df: DataFrame with all lap data
    - weather_df: DataFrame with weather conditions
    - pitstops_df: DataFrame with pit stop information
    """
    print(f"  Downloading FastF1 data for {gp_name} {year}...")
    
    # Get race session
    session = ff1.get_session(year, gp_name, 'R')
    session.load(telemetry = False, weather= True, messages = True)
    
    # Extract data
    laps_df = session.laps.copy()
    weather_df = session.weather_data.copy()
    
    # Get pit stops (available in laps data)
    # Filter laps where PitInTime is not null
    pitstops_df = laps_df[laps_df['PitInTime'].notna()].copy()
    
    print(f"    Laps: {len(laps_df)}, Weather records: {len(weather_df)}, Pit stops: {len(pitstops_df)}")
    
    return laps_df, weather_df, pitstops_df



#### Testing Function with Spain 2023

In [28]:
# Test with Spain 2023 (we know this works from legacy)
print("Testing function with Spain 2023...")
test_laps, test_weather, test_pitstops = download_fastf1_data(2023, 'Spain')
print("Test successful!")


core           INFO 	Loading data for Spanish Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


Testing function with Spain 2023...


req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 1 completed the race distance 00:00.037000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['1', '44', '63', '11', '55', '18', '14', '31', '24', '10', '16', '22', '81', '21', '27', '23', '4', '20', '77', '2']


    Laps: 1312, Weather records: 154, Pit stops: 43
Test successful!


### 4.2 Calculate Basic Feautures

This function is used to add some basic features to the laps dataframe. This new variables are like the ones used in the project's legacy version.

- ``CompoundID`` maps the different tire compounds from strings to numbers from 1 to 5.
- ``TeamID`` maps all the teams to numbers between 1-10.
- ``LapsSincePitStop`` is used as a marker to know how many laps are done with a specific tire set. 
- `FuelLoad:` an estimation of the amount of fuel that the car has.

In [29]:
def calculate_basic_features(laps_df: pd.DataFrame) -> pd.DataFrame:
    """
    Add calculated basic features to laps dataframe.
    
    Features added:
    - CompoundID: Tire compound to integer (1=Soft, 2=Medium, 3=Hard, 4=Inter, 5=Wet)
    - TeamID: Team name to integer (1-10)
    - LapsSincePitStop: Counter since last pit stop per driver
    - FuelLoad: Estimated fuel load (1 - LapNumber/MaxLap)
    
    Parameters:
    - laps_df: DataFrame with lap data
    
    Returns:
    - laps_df: DataFrame with added features
    """
    print("  Calculating basic features...")
    df = laps_df.copy()
    
    # 1. CompoundID
    df['CompoundID'] = df['Compound'].map(COMPOUND_MAPPING)
    # Handle unmapped compounds
    unmapped_compounds = df[df['CompoundID'].isna()]['Compound'].unique()
    if len(unmapped_compounds) > 0:
        print(f"    Warning: Unmapped compounds found: {unmapped_compounds}")
        df['CompoundID'] = df['CompoundID'].fillna(0)
    
    # 2. TeamID
    df['TeamID'] = df['Team'].map(TEAM_MAPPING)
    # Handle unmapped teams
    unmapped_teams = df[df['TeamID'].isna()]['Team'].unique()
    if len(unmapped_teams) > 0:
        print(f"    Warning: Unmapped teams found: {unmapped_teams}")
        df['TeamID'] = df['TeamID'].fillna(0)
    
    # 3. LapsSincePitStop (counter per driver since last pit)
    df = df.sort_values(['DriverNumber', 'LapNumber'])
    df['LapsSincePitStop'] = 0
    
    for driver in df['DriverNumber'].unique():
        driver_mask = df['DriverNumber'] == driver
        driver_laps = df[driver_mask].copy()
        
        # Find pit lap numbers
        pit_laps = driver_laps[driver_laps['PitInTime'].notna()]['LapNumber'].values
        
        for idx in df[driver_mask].index:
            lap_num = df.loc[idx, 'LapNumber']
            if len(pit_laps) == 0:
                df.loc[idx, 'LapsSincePitStop'] = lap_num
            else:
                previous_pits = pit_laps[pit_laps < lap_num]
                if len(previous_pits) > 0:
                    df.loc[idx, 'LapsSincePitStop'] = lap_num - max(previous_pits)
                else:
                    df.loc[idx, 'LapsSincePitStop'] = lap_num
    
    # 4. FuelLoad (simple estimation)
    max_lap = df['LapNumber'].max()
    df['FuelLoad'] = (1 - (df['LapNumber'] / max_lap)).round(4)
    
    print(f"    Features added: CompoundID, TeamID, LapsSincePitStop, FuelLoad")
    
    return df

In [30]:
# Test with Spain 2023 data
print("Testing feature calculation...")
test_laps_processed = calculate_basic_features(test_laps)
print(f"New columns: {[col for col in test_laps_processed.columns if col in ['CompoundID', 'TeamID', 'LapsSincePitStop', 'FuelLoad']]}")
print("Test successful!")

Testing feature calculation...
  Calculating basic features...
    Features added: CompoundID, TeamID, LapsSincePitStop, FuelLoad
New columns: ['CompoundID', 'TeamID', 'LapsSincePitStop', 'FuelLoad']
Test successful!


### 4.3 Find OpenF1 Session Key

OpenF1 uses an unique identifier for each session, so this function queries the session endpoint to find the correct race session key and extract the data correctly.

In [31]:
def find_openf1_session_key(year: int, gp_name: str) -> int:
    """
    Find OpenF1 session_key for a specific race.
    
    OpenF1 API uses session_key as unique identifier for each session.
    This function queries the sessions endpoint to find the Race session key.
    
    Parameters:
    - year: int (2023 or 2024)
    - gp_name: str (e.g., 'Bahrain', 'Spain')
    
    Returns:
    - session_key: int (unique session identifier)
    
    Raises:
    - ValueError: if session not found
    """
    print(f"  Finding OpenF1 session_key for {gp_name} {year}...")
    
    # OpenF1 sessions endpoint
    url = "https://api.openf1.org/v1/sessions"
    
    # Query sessions for the year
    params = {
        'year': year,
        'session_name': 'Race'
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    sessions = response.json()
    
    # Find matching session by location name
    # OpenF1 uses 'location' field (e.g., 'Sakhir' for Bahrain)
    for session in sessions:
        location = session.get('location', '')
        country = session.get('country_name', '')
        
        # Match by location or country name containing gp_name
        if gp_name.lower() in location.lower() or gp_name.lower() in country.lower():
            session_key = session['session_key']
            print(f"    Found session_key: {session_key} (Location: {location})")
            return session_key
    
    # If not found, raise error
    raise ValueError(f"Session not found for {gp_name} {year}")

In [32]:
# Test with Spain 2023
print("Testing OpenF1 session key lookup...")
try:
    test_session_key = find_openf1_session_key(2023, 'Spain')
    print(f"Test successful! Session key: {test_session_key}")
except Exception as e:
    print(f"Test failed: {e}")

Testing OpenF1 session key lookup...
  Finding OpenF1 session_key for Spain 2023...
    Found session_key: 9102 (Location: Barcelona)
Test successful! Session key: 9102


### 4.4 Download OpenF1 Intervals

This function downloads interval data from OpenF1for a specific session. The data returned includes both numeric gaps and lap deficit information, split into separate columns to enable proper analysis:

**Numeric columns** (for analysis):
- `interval_seconds`: Gap to car ahead in seconds
- `gap_to_leader_seconds`: Gap to race leader in seconds

**Categorical columns** (lap deficit info):
- `laps_behind`: Lap deficit to car ahead (e.g., "+1 LAP", "+2 LAPS")
- `gap_to_leader_laps`: Lap deficit to leader

**Strategic features** (calculated here):
- `drs_window`: Boolean flag for DRS activation range (<1.0s, universal FIA threshold)
- `is_lapped`: Boolean flag indicating if car is lapped

**Note:** `undercut_window` is NOT calculated here because pit lane time loss varies significantly by circuit (Monaco ~30s, Monza ~18s). This feature will be calculated in Phase 2.4 with circuit-specific thresholds.

In [33]:
def download_openf1_intervals(session_key: int) -> pd.DataFrame:
    """
    Download interval data from OpenF1 for a specific session.
    
    Returns cleaned interval data with split columns for numeric analysis and categorical info.
    
    Columns returned:
    - interval_seconds: Gap to car ahead in seconds (float, NaN when lapped)
    - laps_behind: Lap deficit to car ahead (None or "+1 LAP", "+2 LAPS")
    - gap_to_leader_seconds: Gap to race leader in seconds (float, NaN when lapped)
    - gap_to_leader_laps: Lap deficit to leader (None or "+1 LAP", "+2 LAPS")
    - drs_window: Boolean indicating if car is within DRS activation range (<1.0s, universal FIA rule)
    - is_lapped: Boolean indicating if car is one or more laps behind leader
    - driver_number: Driver identifier
    - date: Timestamp of measurement
    
    Note: undercut_window NOT calculated here because pit lane time loss varies by circuit.
    This feature will be calculated in Phase 2.4 Feature Engineering with circuit-specific thresholds.
    
    Parameters:
    - session_key: int (OpenF1 session identifier)
    
    Returns:
    - intervals_df: DataFrame with interval data
    """
    print(f"  Downloading OpenF1 intervals (session_key={session_key})...")
    
    # OpenF1 intervals endpoint
    url = "https://api.openf1.org/v1/intervals"
    
    params = {
        'session_key': session_key
    }
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    intervals = response.json()
    intervals_df = pd.DataFrame(intervals)
    
    if len(intervals_df) == 0:
        print("    Warning: No interval data found")
        return intervals_df
    
    # Split interval column: numeric values vs lapped info
    intervals_df['interval_seconds'] = pd.to_numeric(intervals_df['interval'], errors='coerce')
    intervals_df['laps_behind'] = intervals_df['interval'].apply(
        lambda x: x if pd.notna(x) and 'LAP' in str(x).upper() else None
    )
    
    # Split gap_to_leader column: numeric values vs lapped info
    intervals_df['gap_to_leader_seconds'] = pd.to_numeric(intervals_df['gap_to_leader'], errors='coerce')
    intervals_df['gap_to_leader_laps'] = intervals_df['gap_to_leader'].apply(
        lambda x: x if pd.notna(x) and 'LAP' in str(x).upper() else None
    )
    
    # Drop original mixed-type columns
    intervals_df = intervals_df.drop(columns=['interval', 'gap_to_leader'])
    
    # Calculate DRS window (universal 1.0s threshold per FIA regulations)
    intervals_df['drs_window'] = intervals_df['interval_seconds'].apply(
        lambda x: x < 1.0 if pd.notna(x) else False
    )
    
    # Calculate is_lapped flag (objective, circuit-independent)
    intervals_df['is_lapped'] = intervals_df['gap_to_leader_laps'].notna()
    
    print(f"    Intervals: {len(intervals_df)} records")
    print(f"    Features: interval_seconds, gap_to_leader_seconds, drs_window, is_lapped")
    
    return intervals_df


In [34]:
# Test with Spain 2023 session key
print("Testing OpenF1 intervals download...")
test_intervals = download_openf1_intervals(test_session_key)
print(f"Columns: {test_intervals.columns.tolist()[:5]}... (showing first 5)")
print("Test successful!")

Testing OpenF1 intervals download...
    Intervals: 26036 records
    Features: interval_seconds, gap_to_leader_seconds, drs_window, is_lapped
Columns: ['date', 'session_key', 'meeting_key', 'driver_number', 'interval_seconds']... (showing first 5)
Test successful!


In [35]:
def save_race_data(
    year: int,
    gp_name: str,
    laps_df: pd.DataFrame,
    weather_df: pd.DataFrame,
    intervals_df: pd.DataFrame,
    pitstops_df: pd.DataFrame,
    session_key: Optional[int] = None
) -> None:
    """
    Save race data to structured parquet files with metadata.
    
    Output structure:
    data/raw/{year}/{gp_name}/
        - laps.parquet
        - weather.parquet
        - intervals.parquet
        - pitstops.parquet
        - metadata.json
    
    Parameters:
    - year: int
    - gp_name: str
    - laps_df: DataFrame with lap data (with calculated features)
    - weather_df: DataFrame with weather data
    - intervals_df: DataFrame with interval data (already cleaned with split columns)
    - pitstops_df: DataFrame with pit stop data
    - session_key: Optional[int] (OpenF1 session key)
    """
    print(f"  Saving data for {gp_name} {year}...")
    
    # Create directory
    race_path = RAW_DATA_PATH / str(year) / gp_name.replace(' ', '_')
    race_path.mkdir(parents=True, exist_ok=True)
    
    # Save parquet files (no type conversion needed - already handled in download)
    laps_df.to_parquet(race_path / "laps.parquet", index=False)
    weather_df.to_parquet(race_path / "weather.parquet", index=False)
    intervals_df.to_parquet(race_path / "intervals.parquet", index=False)
    pitstops_df.to_parquet(race_path / "pitstops.parquet", index=False)
    
    # Create metadata
    metadata = {
        'year': year,
        'gp_name': gp_name,
        'extraction_date': datetime.now().isoformat(),
        'session_key_openf1': session_key,
        'record_counts': {
            'laps': len(laps_df),
            'weather': len(weather_df),
            'intervals': len(intervals_df),
            'pitstops': len(pitstops_df)
        },
        'calculated_features': {
            'laps': ['CompoundID', 'TeamID', 'LapsSincePitStop', 'FuelLoad'],
            'intervals': ['interval_seconds', 'laps_behind', 'gap_to_leader_seconds', 
                         'gap_to_leader_laps', 'drs_window', 'is_lapped']
        },
        'notes': {
            'drs_window': 'Universal 1.0s threshold per FIA regulations',
            'undercut_window': 'Deferred to Phase 2.4 Feature Engineering (circuit-specific pit lane time loss required)'
        }
    }
    
    # Save metadata
    with open(race_path / "metadata.json", 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"    Saved to: {race_path}")


### 4.5 Save Race Data

This function saves all downloaded and processed data to parquet files in a structured directory format. Each race gets its own folder containing:

- `laps.parquet`: Lap data with calculated features (CompoundID, TeamID, etc.)
- `weather.parquet`: Weather conditions throughout the race
- `intervals.parquet`: Interval data with strategic features (drs_window, is_lapped)
- `pitstops.parquet`: Pit stop information
- `metadata.json`: Race metadata including record counts, calculated features, and notes

The metadata file documents all calculated features and includes notes about strategic decisions (e.g., why undercut_window is deferred to Phase 2.4).

In [36]:
# Test save function
print("Testing save function...")
save_race_data(
    year=2023,
    gp_name='Spain',
    laps_df=test_laps_processed,
    weather_df=test_weather,
    intervals_df=test_intervals,
    pitstops_df=test_pitstops,
    session_key=test_session_key
)
print("Test successful! Check data/raw/2023/Spain/ folder")


Testing save function...
  Saving data for Spain 2023...
    Saved to: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\2023\Spain
Test successful! Check data/raw/2023/Spain/ folder


In [37]:
def validate_race_data(year: int, gp_name: str) -> dict:
    """
    Validate that race data was saved correctly and contains critical columns.
    
    Checks:
    - All required files exist (laps, weather, intervals, pitstops, metadata)
    - Critical columns present in laps data
    - Calculated features present
    
    Parameters:
    - year: int
    - gp_name: str
    
    Returns:
    - validation_report: dict with 'valid' (bool) and 'issues' (list)
    """
    race_path = RAW_DATA_PATH / str(year) / gp_name.replace(' ', '_')
    
    issues = []
    
    # Check files exist
    required_files = ['laps.parquet', 'weather.parquet', 'intervals.parquet', 
                     'pitstops.parquet', 'metadata.json']
    
    for file in required_files:
        if not (race_path / file).exists():
            issues.append(f"Missing file: {file}")
    
    # Check critical columns in laps
    if (race_path / 'laps.parquet').exists():
        laps = pd.read_parquet(race_path / 'laps.parquet')
        
        critical_columns = ['DriverNumber', 'LapNumber', 'LapTime', 'Compound', 
                          'CompoundID', 'TeamID', 'LapsSincePitStop', 'FuelLoad']
        
        missing_cols = [col for col in critical_columns if col not in laps.columns]
        if missing_cols:
            issues.append(f"Missing columns in laps: {missing_cols}")
    
    valid = len(issues) == 0
    
    return {
        'valid': valid,
        'issues': issues,
        'path': str(race_path)
    }

### 4.6 Validate Race Data

This function performs validation checks on saved race data to ensure completeness and data quality. It verifies:

- All required files exist (laps, weather, intervals, pitstops, metadata.json)
- Critical columns are present in laps data (DriverNumber, LapNumber, LapTime, etc.)
- Calculated features are present (CompoundID, TeamID, LapsSincePitStop, FuelLoad)

Returns a validation report with boolean `valid` flag and list of `issues` if any problems are detected.

In [38]:
# Test validation
print("Testing validation function...")
validation_result = validate_race_data(2023, 'Spain')
print(f"Valid: {validation_result['valid']}")
if not validation_result['valid']:
    print(f"Issues: {validation_result['issues']}")
else:
    print("All checks passed!")

Testing validation function...
Valid: True
All checks passed!


---

## Step 5: Main Download Loop Section

In this section we define the main orchestration function that downloads all 46 Grand Prix races from 2023-2024 seasons. This function coordinates all the helper functions defined above and tracks progress with detailed logging.

In [39]:
def download_all_races() -> dict:
    """
    Download all 46 Grand Prix races from 2023-2024 seasons.
    
    Orchestrates the complete download process with progress tracking and error handling.
    
    Returns:
    - download_log: dict with 'successful', 'failed', 'warnings', timestamps
    """
    # Initialize download log
    download_log = {
        'successful': [],
        'failed': [],
        'warnings': [],
        'start_time': datetime.now().isoformat()
    }
    
    print("=" * 60)
    print("STARTING DATA DOWNLOAD")
    print("=" * 60)
    print(f"Total races to download: {len(RACES_2023) + len(RACES_2024)}\n")
    
    # Main download loop with progress bars
    for year in [2023, 2024]:
        races = RACES_2023 if year == 2023 else RACES_2024
        
        print(f"\nYEAR {year} - {len(races)} races")
        
        # Progress bar for this year
        pbar = tqdm(races, desc=f"Downloading {year}", unit="GP")
        
        for gp_name in pbar:
            pbar.set_postfix_str(f"Processing {gp_name}")
            
            try:
                # 1. Download FastF1 data
                laps, weather, pitstops = download_fastf1_data(year, gp_name)
                
                # 2. Calculate basic features
                laps = calculate_basic_features(laps)
                
                # 3. Download OpenF1 intervals (NON-CRITICAL)
                intervals = pd.DataFrame()
                session_key = None
                try:
                    session_key = find_openf1_session_key(year, gp_name)
                    intervals = download_openf1_intervals(session_key)
                except Exception as e:
                    warning_msg = f"{gp_name} {year}: No OpenF1 data"
                    download_log['warnings'].append(warning_msg)
                
                # 4. Apply canonical name alias before saving
                canonical_name = CIRCUIT_NAME_ALIASES.get(gp_name, gp_name)

                # 5. Save all data
                save_race_data(year, canonical_name, laps, weather, intervals, pitstops, session_key)
                
                # 6. Validate saved data
                validation = validate_race_data(year, canonical_name)
                if not validation['valid']:
                    warning_msg = f"{canonical_name} {year}: Validation issues"
                    download_log['warnings'].append(warning_msg)
                
                # Log success
                download_log['successful'].append(f"{canonical_name} {year}")
                pbar.set_postfix_str(f"✓ {canonical_name}")
                
            except Exception as e:
                # Critical error - log and STOP
                error_msg = f"{gp_name} {year}: {str(e)}"
                download_log['failed'].append(error_msg)
                download_log['end_time'] = datetime.now().isoformat()
                
                pbar.close()
                print(f"\nERROR: Download failed for {gp_name} {year}")
                print(f"Error: {str(e)}")
                
                # Save log before stopping
                with open(RAW_DATA_PATH / "download_log.json", 'w') as f:
                    json.dump(download_log, f, indent=2)
                
                print(f"Download log saved to: {RAW_DATA_PATH / 'download_log.json'}")
                raise
        
        pbar.close()
    
    download_log['end_time'] = datetime.now().isoformat()
    print("\nDOWNLOAD COMPLETED SUCCESSFULLY")
    
    return download_log

In [40]:
# Execute main download
download_log = download_all_races()

STARTING DATA DOWNLOAD
Total races to download: 46


YEAR 2023 - 22 races


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


    Laps: 1056, Weather records: 161, Pit stops: 52
  Calculating basic features...
    Features added: CompoundID, TeamID, LapsSincePitStop, FuelLoad
  Finding OpenF1 session_key for Sakhir 2023...
    Found session_key: 7953 (Location: Sakhir)


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

    Laps: 943, Weather records: 151, Pit stops: 61
  Calculating basic features...
    Features added: CompoundID, TeamID, LapsSincePitStop, FuelLoad
  Finding OpenF1 session_key for Lusail 2024...
    Found session_key: 9655 (Location: Lusail)


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count


    Intervals: 20242 records
    Features: interval_seconds, gap_to_leader_seconds, drs_window, is_lapped
  Saving data for Lusail 2024...
    Saved to: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\2024\Lusail


req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']


    Laps: 1035, Weather records: 148, Pit stops: 30
  Calculating basic features...
    Features added: CompoundID, TeamID, LapsSincePitStop, FuelLoad
  Finding OpenF1 session_key for Yas Island 2024...
    Found session_key: 9662 (Location: Yas Island)
    Intervals: 23288 records
    Features: interval_seconds, gap_to_leader_seconds, drs_window, is_lapped
  Saving data for Yas Island 2024...
    Saved to: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\2024\Yas_Island

DOWNLOAD COMPLETED SUCCESSFULLY


### 5.2 Execute Download

Run the main download function to fetch all race data.

### 5.1 Download All Races Function

This function orchestrates the complete download process for all races. It handles:

**Process per race:**
1. Download FastF1 data (laps, weather, pitstops)
2. Calculate basic features (CompoundID, TeamID, LapsSincePitStop, FuelLoad)
3. Download OpenF1 intervals (non-blocking if fails)
4. Save all data to parquet files
5. Validate saved data

**Error handling:**
- FastF1 download fails → Stop execution and report error
- OpenF1 download fails → Continue without intervals (logged as warning)
- Validation fails → Report but continue

**Progress tracking:**
- Uses `tqdm` progress bars for visual feedback
- Download log tracks: successful, failed, warnings
- Log saved to: `data/raw/download_log.json`

**Returns:** Dictionary with download statistics (successful, failed, warnings)

---

## Step 6: Summary Report

Generate comprehensive report of download results and save the complete download log.

In [41]:
def generate_summary_report(download_log: dict) -> None:
    """
    Generate comprehensive summary report of download results.
    
    Displays status overview, data collected statistics, warnings, and failures.
    Saves download log to JSON file for future reference.
    
    Parameters:
    - download_log: dict with download statistics from download_all_races()
    """
    print("=" * 60)
    print("DOWNLOAD SUMMARY REPORT")
    print("=" * 60)
    
    total_races = len(RACES_2023) + len(RACES_2024)
    successful_count = len(download_log['successful'])
    warnings_count = len(download_log['warnings'])
    failed_count = len(download_log['failed'])
    
    print(f"\nStatus Overview:")
    print(f"  Successful: {successful_count}/{total_races}")
    print(f"  Warnings:   {warnings_count}")
    print(f"  Failed:     {failed_count}")
    
    # Calculate total statistics
    total_laps = 0
    total_intervals = 0
    total_pitstops = 0
    
    for year in [2023, 2024]:
        year_path = RAW_DATA_PATH / str(year)
        if year_path.exists():
            for gp_folder in year_path.iterdir():
                if gp_folder.is_dir():
                    metadata_file = gp_folder / "metadata.json"
                    if metadata_file.exists():
                        with open(metadata_file, 'r') as f:
                            metadata = json.load(f)
                            total_laps += metadata['record_counts']['laps']
                            total_intervals += metadata['record_counts']['intervals']
                            total_pitstops += metadata['record_counts']['pitstops']
    
    print(f"\nData Collected:")
    print(f"  Total laps:      {total_laps:,}")
    print(f"  Total intervals: {total_intervals:,}")
    print(f"  Total pitstops:  {total_pitstops:,}")
    
    # Show warnings if any
    if warnings_count > 0:
        print(f"\nWarnings ({warnings_count}):")
        for warning in download_log['warnings']:
            print(f"  - {warning}")
    
    # Show failures if any
    if failed_count > 0:
        print(f"\nFailures ({failed_count}):")
        for failure in download_log['failed']:
            print(f"  - {failure}")
    
    # Save download log
    log_path = RAW_DATA_PATH / "download_log.json"
    with open(log_path, 'w') as f:
        json.dump(download_log, f, indent=2)
    
    print(f"\nDownload log saved to: {log_path}")
    print("\n" + "=" * 60)
    print("REPORT COMPLETE")
    print("=" * 60)

In [42]:
# Generate and display summary report
generate_summary_report(download_log)

DOWNLOAD SUMMARY REPORT

Status Overview:
  Successful: 46/46
  Warnings:   0
  Failed:     0

Data Collected:
  Total laps:      52,340
  Total intervals: 1,111,904
  Total pitstops:  1,835

Download log saved to: c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\raw\download_log.json

REPORT COMPLETE


### 6.2 Generate Report

Run the summary report function to display download statistics.

### 6.1 Generate Summary Report Function

This function generates a comprehensive summary of the download process. It calculates and displays:

**Status metrics:**
- Total successful downloads
- Total warnings (non-critical failures)
- Total critical failures

**Data collected:**
- Total laps across all races
- Total interval records
- Total pit stops

The function also saves the complete download log to `data/raw/download_log.json` for future reference.